# Routing to a Custom Fine-Tuned GGUF Model via Ollama

This cookbook shows how to use LiteLLM to route requests to a locally-deployed, custom fine-tuned GGUF model alongside cloud providers — treating your own model as a first-class citizen in a multi-provider setup.

**Use case**: You have fine-tuned a small model (e.g., Qwen2.5 or Llama-3 base) on domain-specific data, exported it to GGUF, pushed it to Hugging Face Hub, and deployed it locally with Ollama. You want LiteLLM to route certain request types to your fine-tuned model and fall back to a cloud provider for others.

## Prerequisites

```bash
pip install litellm ollama
```

Deploy your GGUF model with Ollama:
```bash
# Pull directly from Hugging Face Hub (no Modelfile needed for basic use)
ollama pull hf.co/your-username/your-model:Q4_K_M

# Create a local alias
ollama cp hf.co/your-username/your-model:Q4_K_M my-model

# Verify it's running
ollama run my-model "Hello"
```

In [ ]:
import litellm
from litellm import completion

# LiteLLM routes to Ollama models using the 'ollama/' prefix.
# Replace 'my-model' with the name you gave your model in 'ollama cp'.

response = completion(
    model="ollama/my-model",
    messages=[{"role": "user", "content": "What is the capital of France?"}],
    api_base="http://localhost:11434",  # default Ollama address
)
print(response.choices[0].message.content)

## Streaming responses from your fine-tuned model

In [ ]:
response = completion(
    model="ollama/my-model",
    messages=[{"role": "user", "content": "Explain quantum entanglement in one paragraph."}],
    api_base="http://localhost:11434",
    stream=True,
)
for chunk in response:
    print(chunk.choices[0].delta.content or "", end="", flush=True)

## Multi-provider routing: local fine-tuned model + cloud fallback

Route domain-specific queries to your fine-tuned model and general queries to a cloud provider.

In [ ]:
import os
from litellm import Router

router = Router(
    model_list=[
        {
            "model_name": "domain-expert",      # logical name your code uses
            "litellm_params": {
                "model": "ollama/my-model",     # your fine-tuned GGUF via Ollama
                "api_base": "http://localhost:11434",
            },
        },
        {
            "model_name": "general",
            "litellm_params": {
                "model": "claude-haiku-4-5",    # cloud fallback
                "api_key": os.environ.get("ANTHROPIC_API_KEY"),
            },
        },
    ],
    routing_strategy="simple-shuffle",
)

# Route to your fine-tuned model
domain_response = router.completion(
    model="domain-expert",
    messages=[{"role": "user", "content": "Query specific to your fine-tuning domain"}],
)
print("Fine-tuned:", domain_response.choices[0].message.content)

# Route to cloud for general tasks
general_response = router.completion(
    model="general",
    messages=[{"role": "user", "content": "Write a haiku about mountains."}],
)
print("Cloud:", general_response.choices[0].message.content)

## Cost tracking for your local model

LiteLLM can track usage for local models too — useful when comparing cost efficiency of your fine-tuned model vs cloud.

In [ ]:
litellm.success_callback = ["langfuse"]  # or any other logger

# Set a custom cost for your local model ($/1M tokens)
litellm.register_model({
    "ollama/my-model": {
        "input_cost_per_token": 0.0,   # local inference = $0 per token
        "output_cost_per_token": 0.0,
        "max_tokens": 8192,
    }
})

response = completion(
    model="ollama/my-model",
    messages=[{"role": "user", "content": "Test"}],
    api_base="http://localhost:11434",
)
print(f"Tokens used: {response.usage.total_tokens}, Cost: ${response._hidden_params.get('response_cost', 0):.6f}")

## Tips for fine-tuned GGUF models

- **Q4_K_M** is the sweet spot for most fine-tuned models: good quality, fast inference, ~40% of full-precision size.
- If your fine-tune used a custom chat template, set it in the Modelfile before running `ollama create`:
  ```dockerfile
  FROM hf.co/your-username/your-model:Q4_K_M
  TEMPLATE "{{ .System }}\n{{ .Prompt }}"
  ```
- Use `ollama list` to confirm the model is registered before routing LiteLLM to it.
- If your model was trained with a specific system prompt, set it via `litellm_params.system_prompt` in the router config.